
# DDAE seismic denoising in PyTorch: supervised synthetic training + unsupervised real-data fine tuning

This notebook re-implements the core idea of the Saad & Chen (2020) **Deep Denoising Autoencoder (DDAE)** in **PyTorch**, following the architecture and workflow used in the original Keras script https://github.com/omarmohamed15/Deep-Denoising-Autoencoder-for-Seismic-Random-Noise-Attenuation/blob/master/DDAE_Denoising.ipynb.

## What this notebook does

1. **Reproduces the original DDAE architecture** with fully connected layers:

   `256 -> 512 -> 256 -> 128 -> 128 -> 256 -> 512 -> 256`

   using ReLU activations on hidden layers and a linear output layer.

2. **Supervised stage on SWAN synthetic post-stack data**
   - trains on noisy/clean synthetic trace windows,
   - evaluates on held-out synthetic samples,
   - visualizes input / denoised / removed-noise / clean target.

3. **Unsupervised stage on SWAN real post-stack data**
   - applies the supervised model directly to real data,
   - then fine-tunes it using the correlation-based loss from the original script,
   - compares **before vs after fine-tuning**.

4. Provides a **clear, documented, runnable example** with:
   - robust data loading helpers,
   - patch extraction / reconstruction,
   - amplitude-safe normalization,
   - publication-style visualizations,
   - optional SNR / correlation metrics.

## Relation to the original script

The original script:
- trains the DDAE **supervised** on synthetic noisy/clean pairs using **MSE**,
- saves the learned weights,
- initializes a second model for **field/real data**,
- fine-tunes it **unsupervised** using a correlation loss that encourages:
  - the denoised output to remain correlated with the input,
  - the removed component `(input - output)` to be weakly correlated with the denoised result. 

This notebook follows the same logic, but adapts it to the **SWAN synthetic and real post-stack `.npz` files**.

## Reference
1. Saad, O.M. and Chen, Y., 2020. Deep denoising autoencoder for seismic random noise attenuation. Geophysics, 85(4), pp.V367-V376.

In [ ]:

# Core imports
import os
import math
import random
from pathlib import Path
from typing import Dict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# Reproducibility
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:

# -------------------------------------------------------------------
# User configuration
# -------------------------------------------------------------------
SYN_PATH = "SWAN_syn_poststack.npz"
REAL_PATH = "SWAN_real_poststack.npz"

TRACE_LEN = 256
LR_SUP = 1e-3
LR_UNSUP = 5e-4

EPOCHS_SUP = 30
EPOCHS_UNSUP = 20
BATCH_SIZE = 128
VAL_FRAC = 0.15

PATCH_LEN = 256
PATCH_STRIDE = 64

MAX_SYN_SAMPLES = 40000
MAX_REAL_PATCHES = 20000

N_SHOW = 4
FIG_DPI = 130


In [ ]:

# -------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------
def list_npz_keys(path: str):
    with np.load(path, allow_pickle=True) as z:
        return list(z.keys())

def _score_key(k, include_terms, exclude_terms=()):
    lk = k.lower()
    score = 0
    for t in include_terms:
        if t in lk:
            score += 3
    for t in exclude_terms:
        if t in lk:
            score -= 2
    return score

def choose_best_key(keys, include_terms, exclude_terms=()):
    scored = [(k, _score_key(k, include_terms, exclude_terms)) for k in keys]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    best_key, best_score = scored[0]
    if best_score <= 0:
        return None
    return best_key

def robust_npz_loader(path: str) -> Dict[str, np.ndarray]:
    with np.load(path, allow_pickle=True) as z:
        return {k: z[k] for k in z.files}

def ensure_2d_traces(a: np.ndarray) -> np.ndarray:
    """
    Convert array to shape (n_traces, trace_len_like).
    For 2D post-stack sections, assume one axis is time/depth and the other is trace index.
    If ndim > 2, flatten all leading dims except the last one when possible.
    """
    a = np.asarray(a)
    if a.ndim == 1:
        return a[None, :]
    if a.ndim == 2:
        if a.shape[0] < a.shape[1]:
            return a.T.copy()
        return a.copy()
    return a.reshape(-1, a.shape[-1])

def standardize_per_trace(x: np.ndarray, eps: float = 1e-8):
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True)
    sd = np.maximum(sd, eps)
    xz = (x - mu) / sd
    return xz, mu, sd

def unstandardize_per_trace(xz: np.ndarray, mu: np.ndarray, sd: np.ndarray):
    return xz * sd + mu

def extract_1d_patches_from_section(section: np.ndarray, patch_len=256, stride=64):
    """
    section: shape (nt, nx) or (nx, nt)
    Returns patches of shape (n_patches, patch_len), metadata for reconstruction.
    We slide along time within each trace.
    """
    sec = np.asarray(section)
    if sec.ndim != 2:
        raise ValueError("section must be 2D")
    if sec.shape[0] < sec.shape[1]:
        sec = sec.T
    nt, nx = sec.shape
    starts = list(range(0, max(1, nt - patch_len + 1), stride))
    if len(starts) == 0 or starts[-1] != nt - patch_len:
        if nt >= patch_len:
            starts.append(nt - patch_len)
        else:
            pad = patch_len - nt
            sec = np.pad(sec, ((0, pad), (0, 0)))
            nt, nx = sec.shape
            starts = [0]

    patches = []
    meta = []
    for ix in range(nx):
        tr = sec[:, ix]
        for s in starts:
            patches.append(tr[s:s+patch_len])
            meta.append((ix, s))
    patches = np.stack(patches, axis=0)
    return patches, meta, sec.shape

def reconstruct_1d_patches_to_section(patches: np.ndarray, meta, sec_shape):
    nt, nx = sec_shape
    acc = np.zeros((nt, nx), dtype=np.float32)
    wgt = np.zeros((nt, nx), dtype=np.float32)
    for p, (ix, s) in zip(patches, meta):
        L = len(p)
        acc[s:s+L, ix] += p
        wgt[s:s+L, ix] += 1.0
    wgt = np.maximum(wgt, 1e-8)
    return acc / wgt

def snr_db(reference, estimate, eps=1e-12):
    num = np.sum(reference**2)
    den = np.sum((reference - estimate)**2) + eps
    return 10.0 * np.log10(num / den)

def corrcoef_torch(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8):
    x = x - x.mean(dim=1, keepdim=True)
    y = y - y.mean(dim=1, keepdim=True)
    num = (x * y).sum(dim=1)
    den = torch.sqrt((x.square().sum(dim=1) * y.square().sum(dim=1)).clamp_min(eps))
    return num / den.clamp_min(eps)

def unsupervised_ddae_loss(y_true: torch.Tensor, y_pred: torch.Tensor,
                           corr1: float = 0.0, corr2: float = 1.0):
    removed = y_true - y_pred
    r  = corr1 - corrcoef_torch(removed, y_pred)
    r1 = corr2 - corrcoef_torch(y_true, y_pred)
    return (r.square() + r1.square()).mean()

def show_three_panel(noisy, denoised, target=None, title_prefix="", aspect="auto", cmap="gray"):
    fig, axes = plt.subplots(1, 4 if target is not None else 3, figsize=(15, 4), dpi=FIG_DPI)
    axes = np.atleast_1d(axes)
    im0 = axes[0].imshow(noisy, aspect=aspect, cmap=cmap)
    axes[0].set_title(f"{title_prefix} Noisy/Input")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(denoised, aspect=aspect, cmap=cmap)
    axes[1].set_title(f"{title_prefix} Denoised")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(noisy - denoised, aspect=aspect, cmap=cmap)
    axes[2].set_title(f"{title_prefix} Removed noise")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    if target is not None:
        im3 = axes[3].imshow(target, aspect=aspect, cmap=cmap)
        axes[3].set_title(f"{title_prefix} Clean target")
        plt.colorbar(im3, ax=axes[3], fraction=0.046)

    for ax in axes:
        ax.set_xlabel("Trace")
        ax.set_ylabel("Time sample")
    plt.tight_layout()
    plt.show()

def show_trace_overlay(noisy, denoised, target=None, trace_idx=None, title="Trace comparison"):
    if noisy.ndim != 2:
        raise ValueError("Expected section/image shape (nt, nx)")
    nt, nx = noisy.shape
    if trace_idx is None:
        trace_idx = nx // 2
    t = np.arange(nt)
    plt.figure(figsize=(6, 4), dpi=FIG_DPI)
    plt.plot(noisy[:, trace_idx], t, label="Noisy", alpha=0.8)
    plt.plot(denoised[:, trace_idx], t, label="Denoised", linewidth=2)
    if target is not None:
        plt.plot(target[:, trace_idx], t, label="Clean target", linewidth=2)
    plt.gca().invert_yaxis()
    plt.title(f"{title} | trace {trace_idx}")
    plt.xlabel("Amplitude")
    plt.ylabel("Time sample")
    plt.legend()
    plt.tight_layout()
    plt.show()



## Original DDAE script: what it is doing

The original implementation uses a **trace-wise fully connected autoencoder**. Each 256-sample seismic trace window is treated as a 1D vector. The network learns a nonlinear map from noisy input to clean output in the synthetic stage, then reuses the same weights as initialization for unsupervised adaptation on real data.

### Synthetic stage
- Load noisy synthetic data `dn` and clean synthetic data `d`.
- Rearrange them into many **256-sample vectors**.
- Train the DDAE using **MSE loss**.

### Field / real-data stage
- Load real noisy data.
- Build the same DDAE architecture.
- Initialize the weights from the synthetic model.
- Fine-tune with a **correlation loss**:
  - maximize correlation between input and denoised output,
  - minimize correlation between denoised output and removed noise `(input - output)`.

That is exactly the logic implemented in the Keras script in https://github.com/omarmohamed15/Deep-Denoising-Autoencoder-for-Seismic-Random-Noise-Attenuation/blob/master/DDAE_Denoising.ipynb, following the paper of "Saad, O.M. and Chen, Y., 2020. Deep denoising autoencoder for seismic random noise attenuation. Geophysics, 85(4), pp.V367-V376".


In [ ]:

# -------------------------------------------------------------------
# PyTorch DDAE model
# -------------------------------------------------------------------
class DDAE(nn.Module):
    '''
    Same dense architecture as the original Keras implementation:
    input -> 512 -> 256 -> 128 -> 128 -> 256 -> 512 -> output
    Hidden activations: ReLU
    Output activation: linear
    '''
    def __init__(self, input_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, input_dim),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:

# -------------------------------------------------------------------
# Dataset utilities
# -------------------------------------------------------------------
class PairDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

class SingleDataset(Dataset):
    def __init__(self, x):
        self.x = torch.as_tensor(x, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i]

def train_supervised(model, train_loader, val_loader=None, epochs=20, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    hist = {"train": [], "val": []}

    for ep in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0
        ntr = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * xb.size(0)
            ntr += xb.size(0)
        tr_loss /= max(ntr, 1)
        hist["train"].append(tr_loss)

        if val_loader is not None:
            model.eval()
            va_loss = 0.0
            nva = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    va_loss += loss.item() * xb.size(0)
                    nva += xb.size(0)
            va_loss /= max(nva, 1)
            hist["val"].append(va_loss)
            print(f"[SUP] Epoch {ep:03d} | train {tr_loss:.6f} | val {va_loss:.6f}")
        else:
            print(f"[SUP] Epoch {ep:03d} | train {tr_loss:.6f}")
    return hist

def fine_tune_unsupervised(model, loader, epochs=10, lr=5e-4, corr1=0.0, corr2=1.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"loss": []}
    for ep in range(1, epochs + 1):
        model.train()
        total = 0.0
        n = 0
        for xb in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = unsupervised_ddae_loss(xb, pred, corr1=corr1, corr2=corr2)
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
            n += xb.size(0)
        total /= max(n, 1)
        hist["loss"].append(total)
        print(f"[UNSUP] Epoch {ep:03d} | loss {total:.6f}")
    return hist

@torch.no_grad()
def predict_array(model, x, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(x), batch_size):
        xb = torch.as_tensor(x[i:i+batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out, axis=0)


In [ ]:

# -------------------------------------------------------------------
# Load synthetic SWAN data
# -------------------------------------------------------------------
assert os.path.exists(SYN_PATH), f"Cannot find {SYN_PATH}. Please place it next to the notebook."

syn = robust_npz_loader(SYN_PATH)
print("Synthetic keys:", list(syn.keys()))

keys = list(syn.keys())
noisy_key = choose_best_key(keys, include_terms=["noisy", "noise", "input", "dn", "patch", "data"],
                            exclude_terms=["clean", "label", "target"])
clean_key = choose_best_key(keys, include_terms=["clean", "label", "target", "gt"],
                            exclude_terms=["noisy", "noise", "input"])

print("Chosen noisy key:", noisy_key)
print("Chosen clean key:", clean_key)

if noisy_key is None or clean_key is None:
    raise RuntimeError(
        "Could not confidently infer noisy/clean keys from the synthetic NPZ file. "
        "Please inspect `list(syn.keys())` and assign them manually."
    )

x_syn_raw = syn[noisy_key]
y_syn_raw = syn[clean_key]

print("Raw synthetic noisy shape:", x_syn_raw.shape)
print("Raw synthetic clean shape:", y_syn_raw.shape)

x_syn = ensure_2d_traces(x_syn_raw)
y_syn = ensure_2d_traces(y_syn_raw)

n = min(len(x_syn), len(y_syn))
x_syn, y_syn = x_syn[:n], y_syn[:n]

def force_trace_len(a, trace_len=256):
    if a.shape[1] == trace_len:
        return a
    if a.shape[1] > trace_len:
        nseg = a.shape[1] // trace_len
        a = a[:, :nseg*trace_len]
        return a.reshape(-1, trace_len)
    pad = trace_len - a.shape[1]
    return np.pad(a, ((0,0),(0,pad)))

x_syn = force_trace_len(x_syn, TRACE_LEN)
y_syn = force_trace_len(y_syn, TRACE_LEN)

if MAX_SYN_SAMPLES is not None:
    n = min(len(x_syn), MAX_SYN_SAMPLES)
    x_syn = x_syn[:n]
    y_syn = y_syn[:n]

x_syn_n, x_mu, x_sd = standardize_per_trace(x_syn)
y_syn_n, y_mu, y_sd = standardize_per_trace(y_syn)

print("Synthetic training samples:", x_syn_n.shape, y_syn_n.shape)


In [ ]:

# -------------------------------------------------------------------
# Supervised training on synthetic data
# -------------------------------------------------------------------
full_ds = PairDataset(x_syn_n, y_syn_n)
n_val = max(1, int(len(full_ds) * VAL_FRAC))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

model_sup = DDAE(input_dim=TRACE_LEN).to(DEVICE)
hist_sup = train_supervised(model_sup, train_loader, val_loader, epochs=EPOCHS_SUP, lr=LR_SUP)

plt.figure(figsize=(6,4), dpi=FIG_DPI)
plt.plot(hist_sup["train"], label="train")
plt.plot(hist_sup["val"], label="val")
plt.title("Supervised synthetic training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# -------------------------------------------------------------------
# Synthetic evaluation and visualization
# -------------------------------------------------------------------
idxs = np.random.choice(len(val_ds), size=min(N_SHOW, len(val_ds)), replace=False)

fig, axes = plt.subplots(len(idxs), 4, figsize=(14, 3*len(idxs)), dpi=FIG_DPI)
if len(idxs) == 1:
    axes = axes[None, :]

snrs_in = []
snrs_out = []

for row, ds_idx in enumerate(idxs):
    xb, yb = val_ds[ds_idx]
    pred = model_sup(xb[None].to(DEVICE)).detach().cpu().numpy()[0]

    noisy = xb.numpy()
    clean = yb.numpy()
    deno  = pred
    noise_removed = noisy - deno

    snr_in = snr_db(clean, noisy)
    snr_out = snr_db(clean, deno)
    snrs_in.append(snr_in)
    snrs_out.append(snr_out)

    for ax, arr, title in zip(
        axes[row],
        [noisy[:, None], deno[:, None], noise_removed[:, None], clean[:, None]],
        [f"Noisy\nSNR={snr_in:.2f} dB", f"Denoised\nSNR={snr_out:.2f} dB", "Removed noise", "Clean target"]
    ):
        im = ax.imshow(arr, aspect="auto", cmap="gray")
        ax.set_title(title)
        ax.set_xlabel("1 trace")
        ax.set_ylabel("Sample")
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Synthetic validation examples", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(f"Mean input SNR : {np.mean(snrs_in):.2f} dB")
print(f"Mean output SNR: {np.mean(snrs_out):.2f} dB")
print(f"Mean SNR gain  : {np.mean(np.array(snrs_out)-np.array(snrs_in)):.2f} dB")


In [ ]:

# -------------------------------------------------------------------
# Load real SWAN post-stack data
# -------------------------------------------------------------------
assert os.path.exists(REAL_PATH), f"Cannot find {REAL_PATH}. Please place it next to the notebook."

real = robust_npz_loader(REAL_PATH)
print("Real keys:", list(real.keys()))

real_keys = list(real.keys())
real_key = choose_best_key(real_keys, include_terms=["real", "data", "section", "noisy", "noise", "dn", "poststack"])
if real_key is None:
    real_key = real_keys[0]
print("Chosen real-data key:", real_key)

real_raw = np.asarray(real[real_key])
print("Raw real-data shape:", real_raw.shape)

if real_raw.ndim == 2:
    real_sec = real_raw.copy()
elif real_raw.ndim == 3:
    real_sec = real_raw[0]
else:
    raise RuntimeError("Expected real data to be a 2D section or a 3D stack of sections.")

if real_sec.shape[0] < real_sec.shape[1]:
    real_sec = real_sec.T.copy()

real_patches, real_meta, real_shape = extract_1d_patches_from_section(real_sec, patch_len=PATCH_LEN, stride=PATCH_STRIDE)
if MAX_REAL_PATCHES is not None:
    real_patches = real_patches[:MAX_REAL_PATCHES]
    real_meta = real_meta[:MAX_REAL_PATCHES]

real_patches_n, real_mu, real_sd = standardize_per_trace(real_patches)

print("Real patches:", real_patches_n.shape)
print("Real section shape used for visualization:", real_shape)


In [ ]:

# -------------------------------------------------------------------
# Apply supervised model directly to real data (before fine tuning)
# -------------------------------------------------------------------
real_pred_before_n = predict_array(model_sup, real_patches_n, batch_size=512)
real_pred_before = unstandardize_per_trace(real_pred_before_n, real_mu, real_sd)

real_deno_before = reconstruct_1d_patches_to_section(real_pred_before, real_meta, real_shape)
real_in_used     = reconstruct_1d_patches_to_section(real_patches,      real_meta, real_shape)

show_three_panel(real_in_used, real_deno_before, target=None, title_prefix="Real before FT")
show_trace_overlay(real_in_used, real_deno_before, target=None, title="Real data before fine tuning")


In [ ]:

# -------------------------------------------------------------------
# Unsupervised fine tuning on real data using the correlation loss
# -------------------------------------------------------------------
model_ft = DDAE(input_dim=TRACE_LEN).to(DEVICE)
model_ft.load_state_dict(model_sup.state_dict())

real_loader = DataLoader(SingleDataset(real_patches_n), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
hist_unsup = fine_tune_unsupervised(model_ft, real_loader, epochs=EPOCHS_UNSUP, lr=LR_UNSUP, corr1=0.0, corr2=1.0)

plt.figure(figsize=(6,4), dpi=FIG_DPI)
plt.plot(hist_unsup["loss"])
plt.title("Unsupervised fine-tuning loss on real data")
plt.xlabel("Epoch")
plt.ylabel("Correlation loss")
plt.tight_layout()
plt.show()


In [ ]:

# -------------------------------------------------------------------
# Compare real-data denoising before vs after unsupervised fine tuning
# -------------------------------------------------------------------
real_pred_after_n = predict_array(model_ft, real_patches_n, batch_size=512)
real_pred_after = unstandardize_per_trace(real_pred_after_n, real_mu, real_sd)

real_deno_after = reconstruct_1d_patches_to_section(real_pred_after, real_meta, real_shape)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=FIG_DPI)

imgs = [
    (real_in_used, "Real input"),
    (real_deno_before, "Direct denoising\n(supervised model only)"),
    (real_in_used - real_deno_before, "Removed noise\n(before FT)"),
    (real_in_used, "Real input"),
    (real_deno_after, "After unsupervised\nfine tuning"),
    (real_in_used - real_deno_after, "Removed noise\n(after FT)")
]

for ax, (arr, title) in zip(axes.ravel(), imgs):
    im = ax.imshow(arr, aspect="auto", cmap="gray")
    ax.set_title(title)
    ax.set_xlabel("Trace")
    ax.set_ylabel("Time sample")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4), dpi=FIG_DPI)
im = plt.imshow(real_deno_after - real_deno_before, aspect="auto", cmap="gray")
plt.title("Change introduced by unsupervised fine tuning")
plt.xlabel("Trace")
plt.ylabel("Time sample")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

show_trace_overlay(real_in_used, real_deno_before, target=None, title="Before fine tuning")
show_trace_overlay(real_in_used, real_deno_after,  target=None, title="After fine tuning")


In [ ]:

# -------------------------------------------------------------------
# Save trained models
# -------------------------------------------------------------------
torch.save(model_sup.state_dict(), "ddae_swan_supervised.pth")
torch.save(model_ft.state_dict(), "ddae_swan_real_finetuned.pth")
print("Saved:")
print(" - ddae_swan_supervised.pth")
print(" - ddae_swan_real_finetuned.pth")



## Notes and practical guidance

### 1. Why this is faithful to the original paper/script
This notebook keeps the same **fully connected DDAE structure** as the original implementation, rather than replacing it with a CNN or U-Net. That is important, because the original method is a **trace-wise dense autoencoder**, not a convolutional image denoiser. 

### 2. Why normalization is done per trace
The original script does not explicitly standardize amplitudes. In PyTorch, training is usually more stable if each trace is standardized. To avoid changing final amplitudes, the notebook stores each trace mean and standard deviation and **reconstructs amplitudes after prediction**.

### 3. What the unsupervised loss is doing
Let:
- `x` = noisy input trace,
- `f(x)` = denoised output,
- `n_hat = x - f(x)` = removed noise.

Then the real-data fine-tuning loss is:

$
\mathcal{L}_{\text{unsup}}
=
\left(c_2 - \rho(x, f(x))\right)^2
+
\left(c_1 - \rho(x-f(x), f(x))\right)^2,
$

where \(\rho(\cdot,\cdot)\) is Pearson correlation.

With the default settings from the script:
- $c_1 = 0$: make removed noise and denoised output weakly correlated,
- $c_2 = 1$: keep denoised output highly correlated with the input.

This follows the same formulation as the original Keras code: https://github.com/omarmohamed15/Deep-Denoising-Autoencoder-for-Seismic-Random-Noise-Attenuation/blob/master/DDAE_Denoising.ipynb.

### 4. If the NPZ keys are not inferred correctly
Print the keys and assign them manually, for example:

```python
x_syn_raw = syn["noisy"]
y_syn_raw = syn["clean"]
real_sec  = real["section_07"]
```

### 5.Homework
For a stronger modern baseline, you could later compare this DDAE against:
- DnCNN / U-Net,
- Noise2Noise / Noise2Self variants,
- DIP-style denoising,
- patchwise 2D CNN denoising on post-stack panels.
